In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#********************INGRESA EL NOMBRE DE ARCHIVO DE ENTRA Y DE SALIDA**********
archivo_entrada = 'w_antihoraria.txt'
archivo_salida = 'w_antihoraria_corregida.txt'


#********************PROCESAMIENTO DE DATOS Y GRAFICACIÓN********************

df = pd.read_csv(archivo_entrada, sep=',', index_col=False, skipinitialspace=True, low_memory=False)

# Seleccionamos (Columna 0 es Tiempo, Columna 7 es Vel. Angular Y)
resultado = df.iloc[:, [0, 7]].copy()
resultado.columns = ['Tiempo', 'Velocidad_Angular_Y_grados']
resultado = resultado.dropna()

# Normalizar y parsear la columna Tiempo
resultado['Tiempo'] = resultado['Tiempo'].astype(str).str.strip()
tiempo_dt = pd.to_datetime(resultado['Tiempo'], format='%H:%M:%S', errors='coerce')
if tiempo_dt.isna().all():
    tiempo_dt = pd.to_datetime(resultado['Tiempo'], errors='coerce')

# Filtrar filas con tiempo inválido
mask_valid = ~tiempo_dt.isna()
resultado = resultado.loc[mask_valid].copy()
tiempo_dt = tiempo_dt.loc[mask_valid]

# Convertir a segundos relativos tomando la primera fila como t=0
base = tiempo_dt.iloc[0]
elapsed = (tiempo_dt - base).dt.total_seconds()
resultado['Tiempo_s'] = elapsed.values

# Convertir velocidad a numérica y a rad/s (suponiendo que está en grados)
resultado['Velocidad_Angular_Y_grados'] = pd.to_numeric(resultado['Velocidad_Angular_Y_grados'], errors='coerce')
resultado = resultado.dropna(subset=['Velocidad_Angular_Y_grados', 'Tiempo_s'])
resultado['Velocidad_Angular_Y_rad_s'] = resultado['Velocidad_Angular_Y_grados'] * (np.pi / 180)

# Reordenar columnas
resultado = resultado[['Tiempo_s', 'Velocidad_Angular_Y_grados', 'Velocidad_Angular_Y_rad_s']]

# Calcular la media y desviación estándar de la tercera columna
media_tercera_columna = resultado['Velocidad_Angular_Y_rad_s'].mean()
std_media = resultado['Velocidad_Angular_Y_rad_s'].std()

# Guardar con separador por espacios
resultado.to_csv(archivo_salida, index=False, sep=' ')

# Graficar la tercera columna en función del tiempo y marcar la media
plt.figure(figsize=(10, 5))
plt.plot(resultado['Tiempo_s'], resultado['Velocidad_Angular_Y_rad_s'], color='steelblue')
plt.axhline(media_tercera_columna, color='crimson', linestyle='--', linewidth=2, label=f'$\\bar{{\\omega}}$ = {media_tercera_columna:.6f} ± {std_media:.6f} rad/s')
plt.xlabel('t (s)')
plt.ylabel('ω (rad/s)')
plt.title('Velocidad angular en función del tiempo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"¡Listo! Guardado en {archivo_salida}")
print(f"$\\bar{{\\omega}}$ = {media_tercera_columna:.6f} ± {std_media:.6f} rad/s")
print(resultado.head(10))

C:\Users\ruizm\AppData\Local\Temp\ipykernel_22784\562810062.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tiempo_dt = pd.to_datetime(resultado['Tiempo'], errors='coerce')


NameError: name 'salida' is not defined